# Mini-Project 1: S3 Data Pipeline

**Goal:** Build an end-to-end data pipeline using S3 patterns (local simulator) that ingests, validates, transforms, and stores ML-ready features.

**Dataset:** 10,000-row synthetic customer dataset with intentional quality issues (null values, duplicates, out-of-range values, mixed-case categoricals).

**Estimated time:** 2-3 hours

**What you'll practice:**
- S3 CRUD operations (put, get, list, delete)
- Data validation and quality checks
- Parquet optimization and storage efficiency
- Pipeline orchestration with error handling and timing
- Pipeline monitoring and run tracking

In [ ]:
import numpy as np
import pandas as pd
import time
import json
import tempfile
from pathlib import Path
import hashlib
from io import BytesIO, StringIO
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shutil
from datetime import datetime

sns.set_style("whitegrid")


class LocalS3Simulator:
    """Simulates AWS S3 operations using the local filesystem."""

    def __init__(self, root_dir=None):
        if root_dir is None:
            self.root_dir = Path(tempfile.mkdtemp(prefix="s3_sim_"))
        else:
            self.root_dir = Path(root_dir)
            self.root_dir.mkdir(parents=True, exist_ok=True)
        self.buckets = {}

    def create_bucket(self, Bucket, **kwargs):
        bucket_path = self.root_dir / Bucket
        bucket_path.mkdir(parents=True, exist_ok=True)
        self.buckets[Bucket] = bucket_path
        return {"ResponseMetadata": {"HTTPStatusCode": 200}}

    def put_object(self, Bucket, Key, Body, **kwargs):
        if Bucket not in self.buckets:
            raise ValueError(f"Bucket '{Bucket}' does not exist.")
        obj_path = self.buckets[Bucket] / Key
        obj_path.parent.mkdir(parents=True, exist_ok=True)
        if isinstance(Body, str):
            Body = Body.encode("utf-8")
        elif isinstance(Body, (pd.DataFrame,)):
            raise TypeError("Body must be bytes or str, not DataFrame.")
        obj_path.write_bytes(Body)
        md5 = hashlib.md5(Body).hexdigest()
        return {"ETag": f'"{md5}"', "ResponseMetadata": {"HTTPStatusCode": 200}}

    def get_object(self, Bucket, Key):
        if Bucket not in self.buckets:
            raise ValueError(f"Bucket '{Bucket}' does not exist.")
        obj_path = self.buckets[Bucket] / Key
        if not obj_path.exists():
            raise FileNotFoundError(f"Object '{Key}' not found in bucket '{Bucket}'.")
        data = obj_path.read_bytes()
        return {
            "Body": BytesIO(data),
            "ContentLength": len(data),
            "ContentType": "application/octet-stream",
            "ETag": f'"{ hashlib.md5(data).hexdigest()}"',
            "ResponseMetadata": {"HTTPStatusCode": 200},
        }

    def list_objects_v2(self, Bucket, Prefix="", MaxKeys=1000):
        if Bucket not in self.buckets:
            raise ValueError(f"Bucket '{Bucket}' does not exist.")
        bucket_path = self.buckets[Bucket]
        contents = []
        for p in sorted(bucket_path.rglob("*")):
            if p.is_file():
                key = str(p.relative_to(bucket_path))
                if key.startswith(Prefix):
                    stat = p.stat()
                    contents.append({
                        "Key": key,
                        "Size": stat.st_size,
                        "LastModified": datetime.fromtimestamp(stat.st_mtime).isoformat(),
                    })
        contents = contents[:MaxKeys]
        return {
            "Contents": contents,
            "KeyCount": len(contents),
            "MaxKeys": MaxKeys,
            "Prefix": Prefix,
            "IsTruncated": False,
            "ResponseMetadata": {"HTTPStatusCode": 200},
        }

    def head_object(self, Bucket, Key):
        if Bucket not in self.buckets:
            raise ValueError(f"Bucket '{Bucket}' does not exist.")
        obj_path = self.buckets[Bucket] / Key
        if not obj_path.exists():
            raise FileNotFoundError(f"Object '{Key}' not found in bucket '{Bucket}'.")
        data = obj_path.read_bytes()
        stat = obj_path.stat()
        return {
            "ContentLength": len(data),
            "ContentType": "application/octet-stream",
            "ETag": f'"{hashlib.md5(data).hexdigest()}"',
            "LastModified": datetime.fromtimestamp(stat.st_mtime).isoformat(),
            "ResponseMetadata": {"HTTPStatusCode": 200},
        }

    def delete_object(self, Bucket, Key):
        if Bucket not in self.buckets:
            raise ValueError(f"Bucket '{Bucket}' does not exist.")
        obj_path = self.buckets[Bucket] / Key
        if obj_path.exists():
            obj_path.unlink()
        return {"ResponseMetadata": {"HTTPStatusCode": 204}}

    def upload_file(self, Filename, Bucket, Key):
        with open(Filename, "rb") as f:
            data = f.read()
        return self.put_object(Bucket=Bucket, Key=Key, Body=data)

    def download_file(self, Bucket, Key, Filename):
        response = self.get_object(Bucket=Bucket, Key=Key)
        Path(Filename).parent.mkdir(parents=True, exist_ok=True)
        with open(Filename, "wb") as f:
            f.write(response["Body"].read())

    def cleanup(self):
        if self.root_dir.exists():
            shutil.rmtree(self.root_dir)
        self.buckets = {}
        print(f"Cleaned up S3 simulator at {self.root_dir}")


print("Imports and LocalS3Simulator loaded.")

## Part 1: Setup and Data Generation

Initialize the S3 simulator and generate a synthetic customer dataset with intentional quality issues.

In [ ]:
# Initialize the S3 simulator and create our bucket
s3 = LocalS3Simulator()
BUCKET = "ml-pipeline-data"
s3.create_bucket(Bucket=BUCKET)
print(f"S3 simulator initialized at: {s3.root_dir}")
print(f"Bucket '{BUCKET}' created.")

In [ ]:
# Generate a 10,000-row synthetic customer dataset with intentional quality issues
np.random.seed(42)
n_rows = 10000

# Base data
customer_ids = [f"CUST-{i:06d}" for i in range(n_rows)]
genders = np.random.choice(["Male", "Female", "MALE", "female", "  Male ", "FEMALE"], n_rows)
senior_citizen = np.random.choice([0, 1], n_rows, p=[0.84, 0.16])
partner = np.random.choice(["Yes", "No", "YES", "no", " Yes"], n_rows)
dependents = np.random.choice(["Yes", "No"], n_rows, p=[0.3, 0.7])

# Tenure with some out-of-range values (-5 to 150)
tenure_months = np.random.normal(32, 20, n_rows).astype(int)
# Inject some out-of-range values
out_of_range_idx = np.random.choice(n_rows, size=int(n_rows * 0.02), replace=False)
tenure_months[out_of_range_idx] = np.random.choice([-5, -3, -1, 125, 130, 150], len(out_of_range_idx))

# Contract and services
contract = np.random.choice(
    ["Month-to-month", "One year", "Two year", "month-to-month", "ONE YEAR"],
    n_rows, p=[0.4, 0.2, 0.2, 0.1, 0.1]
)
internet_service = np.random.choice(["DSL", "Fiber optic", "No"], n_rows, p=[0.35, 0.40, 0.25])
payment_method = np.random.choice(
    ["Electronic check", "Mailed check", "Bank transfer", "Credit card"],
    n_rows
)

# Charges
monthly_charges = np.round(np.random.uniform(18.0, 120.0, n_rows), 2)
total_charges = np.round(monthly_charges * np.maximum(tenure_months, 1) * np.random.uniform(0.9, 1.1, n_rows), 2)

# Churn
churn = np.random.choice(["Yes", "No"], n_rows, p=[0.27, 0.73])

df = pd.DataFrame({
    "customer_id": customer_ids,
    "gender": genders,
    "senior_citizen": senior_citizen,
    "partner": partner,
    "dependents": dependents,
    "tenure_months": tenure_months,
    "contract": contract,
    "internet_service": internet_service,
    "payment_method": payment_method,
    "monthly_charges": monthly_charges,
    "total_charges": total_charges,
    "churn": churn,
})

# Inject ~3% null values in monthly_charges and total_charges
null_idx_monthly = np.random.choice(n_rows, size=int(n_rows * 0.03), replace=False)
null_idx_total = np.random.choice(n_rows, size=int(n_rows * 0.03), replace=False)
df.loc[null_idx_monthly, "monthly_charges"] = np.nan
df.loc[null_idx_total, "total_charges"] = np.nan

# Inject ~2% duplicate rows
n_dupes = int(n_rows * 0.02)
dupe_idx = np.random.choice(n_rows, size=n_dupes, replace=False)
df = pd.concat([df, df.iloc[dupe_idx]], ignore_index=True)

print(f"Dataset shape: {df.shape}")
print(f"\nNull counts:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTenure range: [{df['tenure_months'].min()}, {df['tenure_months'].max()}]")
print(f"\nSample unique genders: {df['gender'].unique()[:6]}")
print(f"\nFirst 5 rows:")
df.head()

## Part 2: Raw Data Ingestion

Upload the raw dataset and metadata to S3, and implement a helper for uploading directories.

In [ ]:
# TODO: Upload the raw CSV to raw/customers/2024-01.csv
# Hint: Convert the DataFrame to CSV bytes, then use s3.put_object()

# YOUR CODE HERE


# ------ Solution (uncomment to use) ------
# csv_buffer = df.to_csv(index=False).encode("utf-8")
# raw_key = "raw/customers/2024-01.csv"
# s3.put_object(Bucket=BUCKET, Key=raw_key, Body=csv_buffer)
# print(f"Uploaded raw CSV to s3://{BUCKET}/{raw_key}")
# head_resp = s3.head_object(Bucket=BUCKET, Key=raw_key)
# print(f"Object size: {head_resp['ContentLength']:,} bytes")

In [ ]:
# TODO: Upload a metadata JSON file with schema, row count, source, and timestamp
# Save to raw/customers/2024-01_metadata.json

# YOUR CODE HERE


# ------ Solution (uncomment to use) ------
# metadata = {
#     "schema": {col: str(dtype) for col, dtype in df.dtypes.items()},
#     "row_count": len(df),
#     "column_count": len(df.columns),
#     "source": "synthetic_generator_v1",
#     "timestamp": datetime.now().isoformat(),
#     "null_counts": df.isnull().sum().to_dict(),
#     "duplicate_count": int(df.duplicated().sum()),
# }
# metadata_key = "raw/customers/2024-01_metadata.json"
# s3.put_object(Bucket=BUCKET, Key=metadata_key, Body=json.dumps(metadata, indent=2))
# print(f"Uploaded metadata to s3://{BUCKET}/{metadata_key}")
# print(json.dumps(metadata, indent=2))

In [ ]:
# TODO: Implement upload_directory() that uploads all files in a local directory to S3
# Signature: upload_directory(s3, bucket, local_dir, s3_prefix)
# It should walk the directory, upload each file, and return a list of uploaded keys.

# YOUR CODE HERE


# ------ Solution (uncomment to use) ------
# def upload_directory(s3_client, bucket, local_dir, s3_prefix):
#     """Upload all files in a local directory to an S3 prefix."""
#     uploaded_keys = []
#     local_path = Path(local_dir)
#     for file_path in local_path.rglob("*"):
#         if file_path.is_file():
#             relative = file_path.relative_to(local_path)
#             s3_key = f"{s3_prefix}/{relative}"
#             s3_client.upload_file(Filename=str(file_path), Bucket=bucket, Key=s3_key)
#             uploaded_keys.append(s3_key)
#             print(f"  Uploaded: {s3_key}")
#     print(f"\nTotal files uploaded: {len(uploaded_keys)}")
#     return uploaded_keys
#
# # Test with a temp directory
# test_dir = Path(tempfile.mkdtemp())
# (test_dir / "file1.txt").write_text("hello")
# (test_dir / "subdir").mkdir()
# (test_dir / "subdir" / "file2.txt").write_text("world")
# uploaded = upload_directory(s3, BUCKET, test_dir, "test-upload")
# print(f"Uploaded keys: {uploaded}")
# # Clean up test
# for key in uploaded:
#     s3.delete_object(Bucket=BUCKET, Key=key)
# shutil.rmtree(test_dir)

In [ ]:
# Verify what's in S3 now
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="raw/")
print(f"Objects under 'raw/' prefix:")
for obj in response.get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']:,} bytes)")

## Part 3: Data Validation

Build a validation function that performs comprehensive data quality checks.

In [ ]:
# TODO: Build validate_data(df) that checks:
#   1. Required columns are present
#   2. Null rates are below threshold (5%)
#   3. Value ranges: tenure 0-120, monthly_charges 0-500, total_charges 0-50000
#   4. Duplicate rate below 1%
#   Returns: dict with "passed" (bool) and "report" (list of check results)

# YOUR CODE HERE


# ------ Solution (uncomment to use) ------
# REQUIRED_COLUMNS = [
#     "customer_id", "gender", "senior_citizen", "partner", "dependents",
#     "tenure_months", "contract", "internet_service", "payment_method",
#     "monthly_charges", "total_charges", "churn",
# ]
#
# RANGE_CHECKS = {
#     "tenure_months": (0, 120),
#     "monthly_charges": (0, 500),
#     "total_charges": (0, 50000),
# }
#
# NULL_THRESHOLD = 0.05  # 5%
# DUPLICATE_THRESHOLD = 0.01  # 1%
#
#
# def validate_data(df):
#     """Validate a DataFrame against quality rules. Returns a report dict."""
#     report = []
#     all_passed = True
#
#     # Check 1: Required columns
#     missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
#     passed = len(missing_cols) == 0
#     report.append({
#         "check": "required_columns",
#         "passed": passed,
#         "detail": f"Missing: {missing_cols}" if not passed else "All required columns present",
#     })
#     all_passed &= passed
#
#     # Check 2: Null rates
#     for col in df.columns:
#         null_rate = df[col].isnull().mean()
#         if null_rate > 0:
#             passed = null_rate <= NULL_THRESHOLD
#             report.append({
#                 "check": f"null_rate_{col}",
#                 "passed": passed,
#                 "detail": f"Null rate: {null_rate:.2%} (threshold: {NULL_THRESHOLD:.0%})",
#             })
#             all_passed &= passed
#
#     # Check 3: Value ranges
#     for col, (low, high) in RANGE_CHECKS.items():
#         if col in df.columns:
#             valid = df[col].dropna()
#             out_of_range = ((valid < low) | (valid > high)).sum()
#             pct = out_of_range / len(valid) if len(valid) > 0 else 0
#             passed = out_of_range == 0
#             report.append({
#                 "check": f"range_{col}",
#                 "passed": passed,
#                 "detail": f"{out_of_range} values out of [{low}, {high}] ({pct:.2%})",
#             })
#             all_passed &= passed
#
#     # Check 4: Duplicate rate
#     dupe_rate = df.duplicated().mean()
#     passed = dupe_rate <= DUPLICATE_THRESHOLD
#     report.append({
#         "check": "duplicate_rate",
#         "passed": passed,
#         "detail": f"Duplicate rate: {dupe_rate:.2%} (threshold: {DUPLICATE_THRESHOLD:.0%})",
#     })
#     all_passed &= passed
#
#     return {"passed": all_passed, "report": report}
#
#
# # Run validation on the raw data
# result = validate_data(df)
# print(f"Validation passed: {result['passed']}\n")
# for item in result["report"]:
#     status = "PASS" if item["passed"] else "FAIL"
#     print(f"  [{status}] {item['check']}: {item['detail']}")

## Part 4: Data Transformation

Clean the data by removing duplicates, imputing nulls, clipping out-of-range values, and standardizing categoricals.

In [ ]:
# TODO: Clean the data:
#   1. Remove duplicate rows
#   2. Impute nulls: median for numeric columns, mode for categorical
#   3. Clip tenure_months to [0, 120], monthly_charges to [0, 500]
#   4. Standardize categorical values: lowercase, strip whitespace
# Return the cleaned DataFrame

# YOUR CODE HERE


# ------ Solution (uncomment to use) ------
# def clean_data(df):
#     """Clean and standardize the customer DataFrame."""
#     df_clean = df.copy()
#     initial_rows = len(df_clean)
#
#     # 1. Remove duplicates
#     df_clean = df_clean.drop_duplicates().reset_index(drop=True)
#     dupes_removed = initial_rows - len(df_clean)
#     print(f"Removed {dupes_removed} duplicate rows ({initial_rows} -> {len(df_clean)})")
#
#     # 2. Impute nulls
#     numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
#     categorical_cols = df_clean.select_dtypes(include=["object"]).columns
#
#     for col in numeric_cols:
#         n_null = df_clean[col].isnull().sum()
#         if n_null > 0:
#             median_val = df_clean[col].median()
#             df_clean[col] = df_clean[col].fillna(median_val)
#             print(f"Imputed {n_null} nulls in '{col}' with median ({median_val:.2f})")
#
#     for col in categorical_cols:
#         n_null = df_clean[col].isnull().sum()
#         if n_null > 0:
#             mode_val = df_clean[col].mode()[0]
#             df_clean[col] = df_clean[col].fillna(mode_val)
#             print(f"Imputed {n_null} nulls in '{col}' with mode ('{mode_val}')")
#
#     # 3. Clip out-of-range values
#     for col, (low, high) in RANGE_CHECKS.items():
#         if col in df_clean.columns:
#             before = ((df_clean[col] < low) | (df_clean[col] > high)).sum()
#             df_clean[col] = df_clean[col].clip(lower=low, upper=high)
#             print(f"Clipped {before} out-of-range values in '{col}' to [{low}, {high}]")
#
#     # 4. Standardize categorical values
#     for col in categorical_cols:
#         if col != "customer_id":  # Don't modify IDs
#             df_clean[col] = df_clean[col].str.strip().str.lower()
#     print(f"\nStandardized categorical columns to lowercase/stripped.")
#
#     print(f"\nFinal shape: {df_clean.shape}")
#     print(f"Remaining nulls: {df_clean.isnull().sum().sum()}")
#     return df_clean
#
#
# df_clean = clean_data(df)

In [ ]:
# TODO: Save the cleaned data as Parquet to processed/customers/2024-01.parquet
# Compare sizes: raw CSV vs Parquet

# YOUR CODE HERE


# ------ Solution (uncomment to use) ------
# parquet_buffer = BytesIO()
# df_clean.to_parquet(parquet_buffer, index=False, engine="pyarrow")
# parquet_bytes = parquet_buffer.getvalue()
#
# processed_key = "processed/customers/2024-01.parquet"
# s3.put_object(Bucket=BUCKET, Key=processed_key, Body=parquet_bytes)
#
# # Compare sizes
# csv_size = s3.head_object(Bucket=BUCKET, Key=raw_key)["ContentLength"]
# parquet_size = s3.head_object(Bucket=BUCKET, Key=processed_key)["ContentLength"]
# print(f"Raw CSV size:        {csv_size:>10,} bytes")
# print(f"Cleaned Parquet size:{parquet_size:>10,} bytes")
# print(f"Size reduction:      {(1 - parquet_size/csv_size)*100:.1f}%")
#
# # Re-validate the cleaned data
# result_clean = validate_data(df_clean)
# print(f"\nCleaned data validation: {'PASSED' if result_clean['passed'] else 'FAILED'}")

## Part 5: Feature Engineering

Create derived features for ML and save them to the features prefix.

In [ ]:
# TODO: Create derived features:
#   1. tenure_bucket: "short" (<12), "medium" (12-36), "long" (>36)
#   2. charge_ratio: monthly_charges / (total_charges / max(tenure_months, 1))
#   3. is_high_value: monthly_charges > 75th percentile
# Save to features/customers/2024-01.parquet

# YOUR CODE HERE


# ------ Solution (uncomment to use) ------
# def engineer_features(df):
#     """Create derived features from cleaned customer data."""
#     df_feat = df.copy()
#
#     # 1. Tenure bucket
#     df_feat["tenure_bucket"] = pd.cut(
#         df_feat["tenure_months"],
#         bins=[-1, 12, 36, 999],
#         labels=["short", "medium", "long"],
#     )
#
#     # 2. Charge ratio
#     avg_monthly = df_feat["total_charges"] / df_feat["tenure_months"].clip(lower=1)
#     df_feat["charge_ratio"] = df_feat["monthly_charges"] / avg_monthly.clip(lower=0.01)
#
#     # 3. Is high value
#     p75 = df_feat["monthly_charges"].quantile(0.75)
#     df_feat["is_high_value"] = (df_feat["monthly_charges"] > p75).astype(int)
#     print(f"75th percentile for monthly_charges: ${p75:.2f}")
#     print(f"High-value customers: {df_feat['is_high_value'].sum()} ({df_feat['is_high_value'].mean():.1%})")
#
#     print(f"\nNew features added: tenure_bucket, charge_ratio, is_high_value")
#     print(f"Feature DataFrame shape: {df_feat.shape}")
#     return df_feat
#
#
# df_features = engineer_features(df_clean)
# df_features[["tenure_months", "tenure_bucket", "monthly_charges",
#              "total_charges", "charge_ratio", "is_high_value"]].head(10)

In [ ]:
# TODO: Save the features DataFrame to S3 under features/ prefix

# YOUR CODE HERE


# ------ Solution (uncomment to use) ------
# feat_buffer = BytesIO()
# df_features.to_parquet(feat_buffer, index=False, engine="pyarrow")
# feat_bytes = feat_buffer.getvalue()
#
# features_key = "features/customers/2024-01.parquet"
# s3.put_object(Bucket=BUCKET, Key=features_key, Body=feat_bytes)
#
# feat_size = s3.head_object(Bucket=BUCKET, Key=features_key)["ContentLength"]
# print(f"Features Parquet size: {feat_size:,} bytes")
# print(f"Saved to s3://{BUCKET}/{features_key}")
#
# # List all objects in the bucket
# print("\nAll objects in bucket:")
# for obj in s3.list_objects_v2(Bucket=BUCKET)["Contents"]:
#     print(f"  {obj['Key']}  ({obj['Size']:,} bytes)")

## Part 6: Pipeline Orchestration

Build a single `run_pipeline()` function that chains Parts 2-5 together with timing and error handling.

In [ ]:
# TODO: Build run_pipeline(s3, bucket, raw_df, run_id) that:
#   1. Ingests raw data (CSV + metadata) to raw/ prefix
#   2. Validates the data; if validation fails, abort and return error
#   3. Cleans/transforms the data
#   4. Engineers features
#   5. Saves all outputs to S3
#   6. Times each stage
#   Returns: dict with status, timing, row counts, S3 keys

# YOUR CODE HERE


# ------ Solution (uncomment to use) ------
# def run_pipeline(s3_client, bucket, raw_df, run_id):
#     """Run the full data pipeline: ingest -> validate -> transform -> features."""
#     result = {
#         "run_id": run_id,
#         "status": "running",
#         "start_time": datetime.now().isoformat(),
#         "stages": {},
#         "row_counts": {},
#         "s3_keys": [],
#     }
#
#     try:
#         # Stage 1: Ingestion
#         t0 = time.time()
#         csv_buffer = raw_df.to_csv(index=False).encode("utf-8")
#         raw_key = f"raw/customers/{run_id}.csv"
#         s3_client.put_object(Bucket=bucket, Key=raw_key, Body=csv_buffer)
#
#         meta = {
#             "run_id": run_id,
#             "row_count": len(raw_df),
#             "timestamp": datetime.now().isoformat(),
#         }
#         meta_key = f"raw/customers/{run_id}_metadata.json"
#         s3_client.put_object(Bucket=bucket, Key=meta_key, Body=json.dumps(meta))
#         result["stages"]["ingestion"] = round(time.time() - t0, 3)
#         result["row_counts"]["raw"] = len(raw_df)
#         result["s3_keys"].extend([raw_key, meta_key])
#         print(f"  [1/4] Ingestion: {result['stages']['ingestion']:.3f}s")
#
#         # Stage 2: Validation
#         t0 = time.time()
#         val_result = validate_data(raw_df)
#         result["stages"]["validation"] = round(time.time() - t0, 3)
#         print(f"  [2/4] Validation: {result['stages']['validation']:.3f}s - "
#               f"{'PASSED' if val_result['passed'] else 'ISSUES FOUND (proceeding with cleaning)'}")
#
#         # Save validation report
#         val_key = f"raw/customers/{run_id}_validation.json"
#         s3_client.put_object(Bucket=bucket, Key=val_key,
#                              Body=json.dumps(val_result, indent=2, default=str))
#         result["s3_keys"].append(val_key)
#
#         # Stage 3: Transformation
#         t0 = time.time()
#         df_cleaned = clean_data(raw_df)
#         parquet_buf = BytesIO()
#         df_cleaned.to_parquet(parquet_buf, index=False, engine="pyarrow")
#         proc_key = f"processed/customers/{run_id}.parquet"
#         s3_client.put_object(Bucket=bucket, Key=proc_key, Body=parquet_buf.getvalue())
#         result["stages"]["transformation"] = round(time.time() - t0, 3)
#         result["row_counts"]["cleaned"] = len(df_cleaned)
#         result["s3_keys"].append(proc_key)
#         print(f"  [3/4] Transformation: {result['stages']['transformation']:.3f}s")
#
#         # Stage 4: Feature engineering
#         t0 = time.time()
#         df_feat = engineer_features(df_cleaned)
#         feat_buf = BytesIO()
#         df_feat.to_parquet(feat_buf, index=False, engine="pyarrow")
#         feat_key = f"features/customers/{run_id}.parquet"
#         s3_client.put_object(Bucket=bucket, Key=feat_key, Body=feat_buf.getvalue())
#         result["stages"]["feature_engineering"] = round(time.time() - t0, 3)
#         result["row_counts"]["features"] = len(df_feat)
#         result["s3_keys"].append(feat_key)
#         print(f"  [4/4] Feature engineering: {result['stages']['feature_engineering']:.3f}s")
#
#         result["status"] = "success"
#         result["total_time"] = round(sum(result["stages"].values()), 3)
#
#     except Exception as e:
#         result["status"] = "failed"
#         result["error"] = str(e)
#         print(f"  Pipeline FAILED: {e}")
#
#     result["end_time"] = datetime.now().isoformat()
#     return result
#
#
# # Test the pipeline
# print("Running pipeline...")
# pipeline_result = run_pipeline(s3, BUCKET, df, "2024-01-test")
# print(f"\nPipeline status: {pipeline_result['status']}")
# print(f"Total time: {pipeline_result.get('total_time', 'N/A')}s")
# print(f"Row counts: {pipeline_result['row_counts']}")

## Part 7: Pipeline Monitoring

Track multiple pipeline runs, display run history, and visualize stage durations.

In [ ]:
# TODO: Run the pipeline 3 times with slightly different data
# Track each run in a pipeline_runs dict (simulating DynamoDB)
# Show run history as a table
# Plot stage durations across runs

# YOUR CODE HERE


# ------ Solution (uncomment to use) ------
# # Simulate a metadata store (like DynamoDB)
# pipeline_runs = {}
#
# # Generate 3 slightly different datasets and run the pipeline on each
# for i in range(3):
#     run_id = f"2024-{i+1:02d}"
#     print(f"\n{'='*60}")
#     print(f"Pipeline Run: {run_id}")
#     print(f"{'='*60}")
#
#     # Generate slightly different data each time
#     np.random.seed(42 + i)
#     n = 10000 + i * 500  # Vary row count slightly
#     run_df = pd.DataFrame({
#         "customer_id": [f"CUST-{j:06d}" for j in range(n)],
#         "gender": np.random.choice(["Male", "Female", "MALE", "female"], n),
#         "senior_citizen": np.random.choice([0, 1], n, p=[0.84, 0.16]),
#         "partner": np.random.choice(["Yes", "No", "YES", "no"], n),
#         "dependents": np.random.choice(["Yes", "No"], n, p=[0.3, 0.7]),
#         "tenure_months": np.clip(np.random.normal(32, 20, n).astype(int), -5, 150),
#         "contract": np.random.choice(["Month-to-month", "One year", "Two year"], n),
#         "internet_service": np.random.choice(["DSL", "Fiber optic", "No"], n),
#         "payment_method": np.random.choice(
#             ["Electronic check", "Mailed check", "Bank transfer", "Credit card"], n
#         ),
#         "monthly_charges": np.round(np.random.uniform(18.0, 120.0, n), 2),
#         "total_charges": np.round(np.random.uniform(100, 8000, n), 2),
#         "churn": np.random.choice(["Yes", "No"], n, p=[0.27, 0.73]),
#     })
#     # Inject nulls and dupes
#     null_idx = np.random.choice(n, size=int(n * 0.03), replace=False)
#     run_df.loc[null_idx, "monthly_charges"] = np.nan
#     dupe_idx = np.random.choice(n, size=int(n * 0.02), replace=False)
#     run_df = pd.concat([run_df, run_df.iloc[dupe_idx]], ignore_index=True)
#
#     result = run_pipeline(s3, BUCKET, run_df, run_id)
#     pipeline_runs[run_id] = result
#
# print(f"\n\nCompleted {len(pipeline_runs)} pipeline runs.")

In [ ]:
# ------ Solution (uncomment to use) ------
# # Display run history table
# history_rows = []
# for run_id, run_data in pipeline_runs.items():
#     history_rows.append({
#         "run_id": run_id,
#         "status": run_data["status"],
#         "raw_rows": run_data["row_counts"].get("raw", "N/A"),
#         "cleaned_rows": run_data["row_counts"].get("cleaned", "N/A"),
#         "feature_rows": run_data["row_counts"].get("features", "N/A"),
#         "total_time_s": run_data.get("total_time", "N/A"),
#         **{f"{stage}_s": t for stage, t in run_data.get("stages", {}).items()},
#     })
#
# history_df = pd.DataFrame(history_rows)
# print("Pipeline Run History:")
# display(history_df)

In [ ]:
# ------ Solution (uncomment to use) ------
# # Plot stage durations across runs
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#
# # Bar chart of stage durations per run
# stages_data = []
# for run_id, run_data in pipeline_runs.items():
#     for stage, duration in run_data.get("stages", {}).items():
#         stages_data.append({"run_id": run_id, "stage": stage, "duration_s": duration})
#
# stages_df = pd.DataFrame(stages_data)
# stages_pivot = stages_df.pivot(index="run_id", columns="stage", values="duration_s")
# stages_pivot.plot(kind="bar", ax=axes[0], rot=0)
# axes[0].set_title("Stage Durations per Pipeline Run")
# axes[0].set_ylabel("Duration (seconds)")
# axes[0].set_xlabel("Run ID")
# axes[0].legend(title="Stage", bbox_to_anchor=(1.0, 1.0))
#
# # Total time per run
# total_times = {run_id: r.get("total_time", 0) for run_id, r in pipeline_runs.items()}
# axes[1].bar(total_times.keys(), total_times.values(), color="steelblue")
# axes[1].set_title("Total Pipeline Duration per Run")
# axes[1].set_ylabel("Duration (seconds)")
# axes[1].set_xlabel("Run ID")
#
# plt.tight_layout()
# plt.show()

## Part 8: Results and Key Findings

Summarize the pipeline results and reflect on what you learned.

In [ ]:
# Compare raw CSV vs processed Parquet sizes
print("S3 Bucket Contents:")
print("=" * 70)
all_objects = s3.list_objects_v2(Bucket=BUCKET)
total_size = 0
for obj in all_objects.get("Contents", []):
    print(f"  {obj['Key']:<50} {obj['Size']:>10,} bytes")
    total_size += obj["Size"]
print(f"{'':50} {'':->10}")
print(f"  {'Total':<50} {total_size:>10,} bytes")

In [ ]:
# Summary table of pipeline metrics
# ------ Solution (uncomment to use) ------
# print("\nPipeline Metrics Summary")
# print("=" * 50)
# for run_id, run_data in pipeline_runs.items():
#     print(f"\nRun: {run_id}")
#     print(f"  Status:           {run_data['status']}")
#     print(f"  Raw rows:         {run_data['row_counts'].get('raw', 'N/A'):,}")
#     print(f"  Cleaned rows:     {run_data['row_counts'].get('cleaned', 'N/A'):,}")
#     print(f"  Feature rows:     {run_data['row_counts'].get('features', 'N/A'):,}")
#     print(f"  Total time:       {run_data.get('total_time', 'N/A')}s")
#     for stage, t in run_data.get('stages', {}).items():
#         print(f"    {stage:<25} {t:.3f}s")

### Key Findings

TODO: Fill in your observations after running the pipeline.

1. **Data quality issues found:**  
   _e.g., ~3% nulls in charges columns, ~2% duplicate rows, out-of-range tenure values..._

2. **Size reduction (CSV vs Parquet):**  
   _e.g., raw CSV was X bytes, Parquet was Y bytes, Z% reduction..._

3. **Pipeline total time:**  
   _e.g., average pipeline run took X seconds across 3 runs..._

4. **Most time-consuming stage:**  
   _e.g., transformation stage took the longest at X seconds because..._

5. **What I'd add for production:**  
   _e.g., retry logic, dead-letter queue for failed records, data lineage tracking, alerting on validation failures, incremental processing..._

In [ ]:
# Cleanup
s3.cleanup()
print("Done! S3 simulator cleaned up.")